# 🧹 Amazon Sales — Data Cleaning Notebook
**Project:** Universal Data Analytics Project  
**Dataset:** Amazon Sales (50,000 orders, Jan 2022 – Dec 2023)  
**Goal:** Produce an analysis-ready `Cleaned_Data.csv` through a *reusable, dataset-agnostic* cleaning pipeline.

### Pipeline Steps
1. Import & Inspect
2. Missing Value Handling
3. Duplicate Removal
4. Data Type Conversion
5. Data Validation & Range Checks
6. Outlier Detection
7. Standardization
8. Feature Engineering
9. Export Cleaned Data

In [ ]:
# --- Imports & Configuration ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

RAW_PATH  = '../Dataset/Raw_Data.csv'
CLEAN_PATH = '../Dataset/Cleaned_Data.csv'
Path('../Dataset').mkdir(parents=True, exist_ok=True)

In [ ]:
# --- 1. Load & Inspect ---
df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Structural overview
print('--- COLUMNS & DTYPES ---')
print(df.dtypes)
print('\n--- NULL COUNTS ---')
print(df.isnull().sum())
print(f'\n--- DUPLICATE ROWS: {df.duplicated().sum()} ---')

## 2. Missing Value Handling
No missing values exist in this dataset, but we keep a **reusable** imputation strategy so the notebook works on *any* dataset:
- **Numeric** → median (robust to outliers)
- **Categorical** → mode
- Drop rows missing critical keys (`order_id`, `order_date`, `total_revenue`)

In [ ]:
# --- 2. Missing Value Handling (reusable strategy) ---
critical_cols = ['order_id', 'order_date', 'total_revenue']
df = df.dropna(subset=critical_cols)

num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include='object').columns

for c in num_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].median())

for c in cat_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].mode()[0])

print('Remaining nulls:', df.isnull().sum().sum())

## 3. Duplicate Removal
Drop fully-duplicate rows and enforce a unique business key on `order_id`.

In [ ]:
# --- 3. Duplicate Removal ---
before = len(df)
df = df.drop_duplicates()
# Enforce unique order_id (keep last occurrence if any collision)
df = df.drop_duplicates(subset=['order_id'], keep='last')
print(f'Removed {before - len(df)} duplicate rows. Rows now: {len(df)}')

## 4. Data Type Conversion
- `order_date` → `datetime`
- IDs → string (avoid arithmetic on identifiers)
- Numerics ensured

In [ ]:
# --- 4. Data Type Conversion ---
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
for c in ['order_id', 'product_id']:
    df[c] = df[c].astype(str)
for c in ['price', 'discounted_price', 'total_revenue', 'rating']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
print(df[['order_id','order_date','price','total_revenue']].dtypes)

## 5. Data Validation & Range Checks
Enforce sensible business boundaries and recompute derived fields so downstream math is always consistent.

In [ ]:
# --- 5. Data Validation & Range Checks ---
rules = {
    'price':            (0, 5000),
    'quantity_sold':    (1, 1000),
    'discount_percent': (0, 100),
    'rating':           (0, 5),
}
for col, (lo, hi) in rules.items():
    bad = ~df[col].between(lo, hi)
    if bad.any():
        print(f'Fixing {bad.sum()} out-of-range {col}')
        df = df[df[col].between(lo, hi)]

# Recompute derived numeric fields for consistency
df['price'] = pd.to_numeric(df['price'])
df['discounted_price'] = (df['price'] * (1 - df['discount_percent']/100)).round(2)
df['total_revenue'] = (df['discounted_price'] * df['quantity_sold']).round(2)
print('Validation complete. Rows:', len(df))

## 6. Outlier Detection
Use the **IQR method** on `total_revenue`. We *flag* (not blindly drop) so business context is preserved. Extreme transactions are reviewed but retained here because high-value orders are legitimate in e-commerce.

In [ ]:
# --- 6. Outlier Detection (IQR) ---
def detect_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return (series < lower) | (series > upper), lower, upper

mask, lo, hi = detect_outliers_iqr(df['total_revenue'])
print(f'IQR bounds: [{lo:.2f}, {hi:.2f}]')
print(f'Flagged outliers: {mask.sum()} ({mask.mean()*100:.2f}%)')

fig, ax = plt.subplots(1, 2, figsize=(13,4))
sns.boxplot(x=df['total_revenue'], ax=ax[0], color='#FF9900').set_title('Total Revenue — Boxplot (with outliers)')
sns.histplot(df['total_revenue'], bins=50, ax=ax[1], color='#146EB4').set_title('Total Revenue — Distribution')
plt.tight_layout(); plt.show()

## 7. Standardization
Standardise text casing/whitespace in categorical fields.

In [ ]:
# --- 7. Standardization ---
text_cols = ['product_category', 'customer_region', 'payment_method']
for c in text_cols:
    df[c] = df[c].astype(str).str.strip().str.title()
# Restore exact category names
df['product_category'] = df['product_category'].replace({'Home & Kitchen':'Home & Kitchen'})
print(df[text_cols].apply(lambda s: s.unique()[:8]))

## 8. Feature Engineering
Create profit (category-based margin), cost, and rich date dimensions for trend/segment analysis.

In [ ]:
# --- 8. Feature Engineering ---
# Profit: assumed category-level gross margin (business assumption — adjustable)
margin_map = {
    'Electronics': 0.28, 'Fashion': 0.40, 'Beauty': 0.45,
    'Books': 0.35, 'Sports': 0.30, 'Home & Kitchen': 0.25,
}
df['profit_margin'] = df['product_category'].map(margin_map).fillna(0.30)
df['profit'] = (df['total_revenue'] * df['profit_margin']).round(2)
df['cost']   = (df['total_revenue'] - df['profit']).round(2)

# Date dimensions
df['year']       = df['order_date'].dt.year
df['month']      = df['order_date'].dt.month
df['year_month'] = df['order_date'].dt.to_period('M').astype(str)
df['quarter']    = df['order_date'].dt.quarter
df['day_of_week']= df['order_date'].dt.day_name()
df['is_weekend'] = df['order_date'].dt.dayofweek >= 5

# Business segments
df['discount_band'] = pd.cut(df['discount_percent'], bins=[-1,0,10,20,100],
                             labels=['No Discount','Low (1-10%)','Medium (11-20%)','High (21%+)'])
df['rating_tier']   = pd.cut(df['rating'], bins=[0,2,3,4,5],
                             labels=['Poor','Average','Good','Excellent'])
df['aov'] = df['total_revenue']  # per-order value

print('New columns added. Final shape:', df.shape)
df.head()

## 9. Export Cleaned Data

In [ ]:
# --- 9. Export ---
df.to_csv(CLEAN_PATH, index=False)
print(f'✅ Cleaned data saved → {CLEAN_PATH}')
print(f'   Rows: {len(df):,} | Columns: {df.shape[1]}')
print('\nFinal column list:')
for c in df.columns: print('  •', c)